In [1]:
pip install --upgrade gradio

  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached fastapi-0.115.12-py3-none-any.whl.metadata (27 kB)
  Using cached ffmpy-0.5.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached huggingface_hub-0.30.2-py3-none-any.whl.metadata (13 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.20-py3-none-any.whl.metadata (1.8 kB)
  Using cached safehttpx-0.1.6-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-0.46.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached tomlkit-0.13.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached uvicorn-0.34.2-py3-none-any.whl.metadata (6.5 kB)
  Using cached fsspec-2025.3.2-py3-none-any.whl.metadata (11 kB)
  Using cached websockets-15.0.1-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_6

In [2]:
pip install --upgrade openai

  Using cached jiter-0.9.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.0/662.0 kB 7.0 MB/s eta 0:00:00
Using cached jiter-0.9.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (351 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import time
import gradio as gr

#edllmgpt4turbo
API_KEY = "-"
AZURE_ENDPOINT = "-"


TEXT_PROMPT = {
    "Data 100" : "You are acting as a tutor to computer science and data science students. DO NOT DIRECTLY GIVE THE ANSWER! ASK FOR UNDERSTANDING! First, ask what class they are working on: CS61A or Data 100. Act accordingly to the reponse. Help the students get to the answer on their own without explicitly giving the answer! ACT AS A TUTOR. AVOID GIVING EXAMPLES. MAKE THE STUDENTS GET TO THE ANSWER WITH GUIDING QUESTIONS",
    "CS 61A" : "You are acting as a tutor to computer science and data science students. DO NOT DIRECTLY GIVE THE ANSWER! ASK FOR UNDERSTANDING! First, ask what class they are working on: CS61A or Data 100. Act accordingly to the reponse. Help the students get to the answer on their own without explicitly giving the answer! ACT AS A TUTOR. AVOID GIVING EXAMPLES. MAKE THE STUDENTS GET TO THE ANSWER WITH GUIDING QUESTIONS"
}


# Welcome! My name is Edison Message
js = """
function createGradioAnimation() {
    var container = document.createElement('div');
    container.id = 'gradio-animation';
    container.style.fontSize = '2em';
    container.style.fontWeight = 'bold';
    container.style.textAlign = 'center';
    container.style.marginBottom = '20px';

    var text = 'Welcome! My name is Edison.';
    for (var i = 0; i < text.length; i++) {
        (function(i){
            setTimeout(function(){
                var letter = document.createElement('span');
                letter.style.opacity = '0';
                letter.style.transition = 'opacity 0.5s';
                letter.innerText = text[i];

                container.appendChild(letter);

                setTimeout(function() {
                    letter.style.opacity = '1';
                }, 50);
            }, i * 250);
        })(i);
    }

    var gradioContainer = document.querySelector('.gradio-container');
    gradioContainer.insertBefore(container, gradioContainer.firstChild);

    return 'Animation created';
}
"""


"""
Function: azure_openai_response
Parameters: message: latest user input, history: prev interactions
Returns: the model's response
"""
#remove class_input if take away drop down menu
def azure_openai_response(message, history, system_prompt, class_input):
    #print(class_input)
    # dictionary passing data through JSON file to the API
    headers = {
        'Content-Type': 'application/json',
        'api-key': API_KEY,
    }
    # format history to include only valid role-content dictionaries
    formatted_history = [
        {"role": entry['role'], "content": entry['content']}
        for entry in history if isinstance(entry, dict) and 'role' in entry and 'content' in entry
    ]

    # add system prompt at the beginning of the conversation
    if system_prompt and len(formatted_history) == 0:
        formatted_history.insert(0, {"role": "system", "content": system_prompt})
    
    # add the user's latest message to the history
    formatted_history.append({"role": "user", "content": message})

    # payload for the API
    # CHANGE MESSAGE LIMIT OF API HERE 
    payload = {
        "messages": formatted_history,
        "max_tokens": 400, # CHANGE THIS IF WANT LONGER OUTPUT MESSAGES FROM THE API 
        "temperature": 0.7,
    }

    # send the request to the Azure OpenAI endpoint
    response = requests.post(AZURE_ENDPOINT, json=payload, headers=headers)

    # If success
    if response.status_code == 200:  # Success
        assistant_response = response.json()['choices'][0]['message']['content'].strip()
        ## uncomment below code when remove streaming
        #return {"role": "assistant", "content": assistant_response}
        
        ## streaming code
        # simulate streaming by splitting the response into characters
        output_text = ""
        for char in assistant_response:
            output_text += char
            # Use Gradio's update function to incrementally update the assistant's response
            yield gr.update(text=output_text)
            time.sleep(0.05)  # delay for character-by-character output (adjust timing if needed)
        ## end of streaming code 
    # If fail
    else:  # error handling
        return [{"role": "user", "content": message}, {"role": "assistant", "content": f"Error: {response.status_code} - {response.text}"}]

# create interface 
# number of parameters has to equal number of parameters in azure_openai_response
# remove gr.dropdown from additional_inputs if dont want it 
interface = gr.ChatInterface(
                fn = azure_openai_response, 
                type = 'messages',
                # uncomment below for dropdown menu and to pass in prompt 
                # additional_inputs = [gr.Textbox(TEXT_PROMPT, label="System Prompt", visible = False), gr.Dropdown(["Data 100", "CS 61A"], label="Class", info="Enter the class you are asking about")]
                additional_inputs = [gr.Textbox(TEXT_PROMPT, label="System Prompt", visible = False)] 
)


#interface.launch(share = True)
with gr.Blocks(js=js, theme=gr.themes.Default()) as block:
    # Add markdown to the interface 
    #gr.Markdown("## Welcome. My name is Edison.")
    #gr.Markdown("### Ask me questions about class material, homework, or a project, and I will try to lead you to the answer!")
    interface.render()
    block.load()

block.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://24c323ffdf11bfbfad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
